<a href="https://www.kaggle.com/code/mobeenfatimah/smart-factory-end-to-end-predictive-pipeline?scriptVersionId=349714375" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

#  INDUSTRIAL SMART FACTORY PREDICTIVE MAINTENANCE
### **Interactive Machine Learning Pipeline**

> Predict machine failure risk, identify operational bottlenecks, and perform dynamic inference using tree-based gradient boosting models built on a high-dimensional Industrial IoT dataset.

---

### **Execution Pipeline Architecture**
`Setup` ➔ `Load Data` ➔ `Health Check` ➔ `EDA` ➔ `Correlations` ➔ `Data Split` ➔ `Benchmark` ➔ `Tuning` ➔ `Confusion Matrix` ➔ `Importance` ➔ `Refit` ➔ `Inference` ➔ `Export Model`

---

### **Table of Contents**
1. **Setup** — Lightweight but Complete
2. **Load the Kaggle Dataset**
3. **Dataset Health Check**
4. **Target Identification & EDA**
5. **Feature Relationships**
6. **Data Preparation & Leakage-Safe Split**
7. **Fast Model Benchmark**
8. **Select Best Model & Retrain Pipeline**
9. **Confusion Matrix Evaluation**
10. **Permutation Feature Importance**
11. **Refit on Complete Dataset**
12. **Create Kaggle Test Predictions**
13. **Interactive Delivery / Failure Predictor**
14. **Save Trained Model**
15. **Final Model Card**

## 1. Setup

In [ ]:
# ==============================================================================
# INDUSTRIAL SMART FACTORY PREDICTIVE MAINTENANCE
# Interactive Machine Learning Pipeline
# ==============================================================================
"""
EXECUTION PIPELINE ARCHITECTURE:
Setup -> Load Data -> Health Check -> EDA -> Correlations -> Data Split -> Benchmark -> Tuning
Confusion Matrix -> Importance -> Refit -> Inference -> Export Model
"""

# ==============================================================================
# 1. Setup
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import ipywidgets as widgets
from IPython.display import display, HTML

# Dark Theme Configuration
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0A0D10'
plt.rcParams['axes.facecolor'] = '#121A2F'
plt.rcParams['axes.edgecolor'] = '#38BDF8'
plt.rcParams['axes.labelcolor'] = '#FFFFFF'
plt.rcParams['xtick.color'] = '#E2E8F0'
plt.rcParams['ytick.color'] = '#E2E8F0'
plt.rcParams['grid.color'] = '#1E293B'
plt.rcParams['grid.alpha'] = 0.5

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment Ready")

## 2. Load the Kaggle Dataset

In [ ]:
# ==============================================================================
# 2. Load the Kaggle Dataset
# ==============================================================================
DATASET_PATH = "/kaggle/input/datasets/mobeenfatimah/smart-factory-predictive-maintenance-dataset/"

csv_files = glob.glob(os.path.join(DATASET_PATH, "*.csv")) + glob.glob("./*.csv") + glob.glob("/kaggle/input/*/*.csv")

if csv_files:
    data_file = csv_files[0]
    df = pd.read_csv(data_file)
    print(f"Dataset successfully loaded from: {data_file}")
else:
    raise FileNotFoundError("Dataset path not found. Please attach the dataset to your Kaggle environment.")

print(f"Data Matrix Dimensions: {df.shape[0]:,} Rows | {df.shape[1]} Columns")

## 3. Dataset Health Check

In [ ]:
# ==============================================================================
# 3. Dataset Health Check
# ==============================================================================
health_summary = pd.DataFrame({
    'Data_Type': df.dtypes,
    'Missing_Values': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df)) * 100
})

print("\n==== Data Integrity Summary ====")
display(health_summary.head(15))
print(f"Duplicate Row Count: {df.duplicated().sum()}")

## 4. Target Identification & EDA

In [ ]:
# ==============================================================================
# 4. Target Identification & EDA
# ==============================================================================
TARGET_COL = 'machine_failure'

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x=TARGET_COL, palette=['#38BDF8', '#F43F5E'], ax=ax)
ax.set_title(f"Target Class Distribution ({TARGET_COL})", fontsize=12, fontweight='bold', color='#38BDF8')
ax.set_xlabel("Machine Failure (0 = Normal, 1 = Failure)", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Feature Relationships

In [ ]:
# ==============================================================================
# 5. Feature Relationships
# ==============================================================================
numeric_cols = df.select_dtypes(include=[np.number]).columns

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=False, fmt='.2f', cmap='mako', ax=ax, linewidths=0.1)
ax.set_title("Feature Inter-Correlation Matrix", fontsize=12, fontweight='bold', color='#38BDF8')
plt.tight_layout()
plt.show()

## 6. Data Preparation & Leakage-Safe Split

In [ ]:
# ==============================================================================
# 6. Data Preparation & Leakage-Safe Split
# ==============================================================================
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Drop unique identifier columns to prevent data leakage
id_cols = [c for c in X.columns if 'id' in c.lower() or 'timestamp' in c.lower() or 'date' in c.lower()]
if id_cols:
    X = X.drop(columns=id_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training Matrix Shape : {X_train.shape[0]:,} records | {X_train.shape[1]} features")
print(f"Testing Matrix Shape  : {X_test.shape[0]:,} records | {X_test.shape[1]} features")

## 7. Fast Model Benchmark

In [ ]:
# ==============================================================================
# 7. Fast Model Benchmark
# ==============================================================================
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

models = {
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)
}

benchmark_results = []

for name, model_inst in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model_inst)
    ])
    
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    benchmark_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1-Score': f1_score(y_test, preds, zero_division=0)
    })

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

## 8. Select Best Model & Retrain Pipeline

In [ ]:
# ==============================================================================
# 8. Select Best Model & Retrain Pipeline
# ==============================================================================
best_model_name = benchmark_df.sort_values(by='F1-Score', ascending=False).iloc[0]['Model']
print(f"Selected Optimal Model: {best_model_name}")

best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', models[best_model_name])
])

best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

## 9. Confusion Matrix Evaluation

In [ ]:
# ==============================================================================
# 9. Confusion Matrix Evaluation
# ==============================================================================
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_title(f"Confusion Matrix ({best_model_name})", fontsize=12, fontweight='bold', color='#38BDF8')
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

## 10. Permutation Feature Importance

In [ ]:
# ==============================================================================
# 10. Permutation Feature Importance
# ==============================================================================
perm_imp = permutation_importance(best_pipeline, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)

sorted_idx = perm_imp.importances_mean.argsort()[-15:]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(X_test.columns[sorted_idx], perm_imp.importances_mean[sorted_idx], color='#38BDF8', edgecolor='#0284C7')
ax.set_title("Permutation Feature Importance (Top 15)", fontsize=12, fontweight='bold', color='#38BDF8')
ax.set_xlabel("Mean Score Decrease")
plt.tight_layout()
plt.show()

## 11. Refit on Complete Dataset

In [ ]:
# ==============================================================================
# 11. Refit on Complete Dataset
# ==============================================================================
best_pipeline.fit(X, y)
print("Pipeline successfully refitted on full dataset.")

## 12. Create Kaggle Test Predictions

In [ ]:
# ==============================================================================
# 12. Create Kaggle Test Predictions
# ==============================================================================
mock_test = X_test.copy().reset_index(drop=True)
mock_preds = best_pipeline.predict(mock_test)

submission_df = pd.DataFrame({
    'Order_Id': np.arange(len(mock_preds)),
    'Predicted_Delivery_Status': mock_preds
})

submission_path = "predictive_maintenance_predictions.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission predictions written to: {submission_path}")

## 13. Interactive Delivery / Failure Predictor

In [ ]:
# ==============================================================================
# 13. Interactive Delivery / Failure Predictor
# ==============================================================================
feature_names = X.columns.tolist()
input_widgets = {}

for col in feature_names[:6]:  # Display top features for dynamic input
    if np.issubdtype(X[col].dtype, np.number):
        input_widgets[col] = widgets.FloatSlider(
            value=float(X[col].mean()),
            min=float(X[col].min()),
            max=float(X[col].max()),
            step=float((X[col].max() - X[col].min()) / 50),
            description=col[:15]
        )
    else:
        unique_vals = X[col].dropna().unique().tolist()
        input_widgets[col] = widgets.Dropdown(
            options=unique_vals,
            value=unique_vals[0],
            description=col[:15]
        )

predict_btn = widgets.Button(description="Predict Health Status", button_style="primary", layout=widgets.Layout(width="200px"))
output_box = widgets.Output()

def on_click(b):
    with output_box:
        output_box.clear_output()
        sample_data = {c: [input_widgets[c].value] if c in input_widgets else [X[c].mode()[0]] for c in feature_names}
        input_df = pd.DataFrame(sample_data)
        pred = best_pipeline.predict(input_df)[0]
        
        status = "CRITICAL: FAILURE RISK" if pred == 1 else "OPERATIONAL: NORMAL"
        color = "#F43F5E" if pred == 1 else "#10B981"
        
        display(HTML(f"""
        <div style="background:#131B2F; border:1px solid {color}; padding:12px; border-radius:8px; margin-top:10px; width:280px; text-align:center;">
            <span style="color:#94A3B8; font-size:11px;">PREDICTED STATUS</span><br>
            <h3 style="color:{color}; margin:4px 0 0 0;">{status}</h3>
        </div>
        """))

predict_btn.on_click(on_click)

# ✅ Fixed: Changed HTML(...) to widgets.HTML(...)
ui = widgets.VBox([
    widgets.HTML("<h4 style='color:#38BDF8;'>Interactive Predictor Widget</h4>"),
    widgets.GridBox(list(input_widgets.values()), layout=widgets.Layout(grid_template_columns="repeat(2, 300px)")),
    predict_btn,
    output_box
])

display(ui)

## 14. Save Trained Model

In [ ]:
# ==============================================================================
# 14. Save Trained Model
# ==============================================================================
model_path = "predictive_maintenance_pipeline.joblib"
joblib.dump(best_pipeline, model_path)
print(f"Model saved to '{model_path}'")

## 15. Final Model Card

In [ ]:
# ==============================================================================
# 15. Final Model Card
# ==============================================================================
model_card = pd.DataFrame([
    {"Attribute": "Task Domain", "Value": "Industrial IoT & Predictive Maintenance"},
    {"Attribute": "Dataset", "Value": "Smart Factory Predictive Maintenance Dataset"},
    {"Attribute": "Total Samples", "Value": f"{len(df):,}"},
    {"Attribute": "Selected Estimator", "Value": best_model_name},
    {"Attribute": "Artifact Location", "Value": model_path}
])

display(model_card)

### **Executive Conclusion**
The gradient boosting pipeline successfully completed multi-model evaluation, permutation feature ranking, and refitting. Automated output and prediction files are formatted and saved in the workspace.

---

### **License & Continuation**
This notebook has been released under the **Apache 2.0 open source license**.